# Pythonic Code

## Creating your own sequences: Composition vs. Inheritance

**The Golden Rule**: Favor Composition ("Has-a") over Inheritance ("Is-a").

**What problem it solves**: Inheriting from standard library objects (like UserList) gives your custom class every single list method (.clear(), .pop()). This is dangerous for domain objects because other developers can bypass your business rules.

**The Solution**: Wrap a list privately (Composition) and inherit from an Abstract Base Class (ABC) like Sequence to selectively expose safe, read-only behaviors (like len() and for loops).

In [1]:
from collections.abc import Sequence

# By inheriting from Sequence, we declare our intention: "This acts like a read-only list"
class ShoppingCart(Sequence):
    def __init__(self):
        # Composition: The real list is hidden inside
        self._items = []
        self.total_price = 0.0

    # Business rules are strictly protected
    def add_item(self, item: str, price: float):
        self._items.append(item)
        self.total_price += price

    # Sequence requirement 1: Allows len(cart)
    def __len__(self) -> int:
        return len(self._items)

    # Sequence requirement 2: Allows cart[0] AND for-loops
    def __getitem__(self, index: int) -> str:
        return self._items[index]

# Usage: Perfectly safe, perfectly Pythonic
cart = ShoppingCart()
cart.add_item("Monitor", 300.0)

for item in cart:  # Works for free!
    print(item)
    
# cart.clear() -> Will throw an AttributeError. Business rules protected!

Monitor


## Context Managers: The @contextmanager Decorator

**What it is**: A decorator from the contextlib module that turns a standard generator function (using the yield keyword) into a fully functional context manager, replacing the need to write a class with `__enter__` and `__exit__` methods.

**What problems it solves**:

Reduces Boilerplate: Eliminates bulky class definitions for simple setup/teardown tasks.

The "Lost Object" Problem: Unlike `@ContextDecorator`, this approach allows you to `yiel` the created resource (like a database connection) so the caller can actually interact with it using with ... as ....

**Execution Safety**: By wrapping the `yield` statement in a try/finally block, it guarantees that resources are correctly closed even if the application crashes.

**When to use it**: Whenever you open a temporary resource that must be safely closed (database sessions, file streams, network sockets) or when building dependencies in modern web frameworks like FastAPI.

In [ ]:
import contextlib
import logging

@contextlib.contextmanager
def safe_database_session():
    # 1. SETUP (This replaces __enter__)
    print("Opening database connection...")
    session = "FastAPI_DB_Session_Object"
    
    try:
        # 2. YIELD THE RESOURCE
        # The 'with' block executes exactly at this line.
        # If the 'with' block crashes, the exception is thrown right here.
        yield session
        
    except ValueError as e:
        # 3. ERROR HANDLING
        # Catch specific, expected errors to recover gracefully.
        logging.error(f"Invalid data provided: {e}. Rolling back transaction!")
        
    except Exception as e:
        # Catch unexpected errors, log them, and re-raise them.
        logging.critical("Unexpected crash! Rolling back transaction!")
        raise e
        
    finally:
        # 4. TEARDOWN (This replaces __exit__)
        # This ALWAYS executes, preventing memory leaks and locked databases.
        print("Closing database connection safely.\n")

# --- Usage Example ---
with safe_database_session() as db:
    print(f"Running queries using: {db}")
    # If we raise a ValueError here, the context manager's 'except' block catches it!

## Assignment Expressions: The Walrus Operator ( `:=` )

**What it is**: Introduced in Python 3.8, the Walrus operator ( `:=` ) allows you to assign a value to a variable and return that value in the exact same expression.

**What problem it solves**: It eliminates redundant code and prevents double-execution of slow functions. Before the Walrus operator, you often had to calculate a value on one line, and then evaluate it on the next (the "calculate, then check" pattern).

**When to use it**:

Inside `while` loops (like reading files chunk-by-chunk or receiving network data).

Inside list comprehensions when you need to filter based on a heavy function, but also want to keep the result of that function in the final list.

**Clean Code Warning**: Variables assigned using `:=` inside a list comprehension "leak" into your surrounding local scope (unlike normal comprehension variables which are safely isolated). Furthermore, do not abuse it to create massive, unreadable one-liners. If a Walrus operator makes the code harder to read, use standard assignment on two lines instead.

In [ ]:
import time

def slow_database_query(user_id: int) -> int:
    """Simulates a heavy function that takes time to run."""
    time.sleep(0.5) 
    return user_id * 10

users = [1, 2, 3, 4]

# ---------------------------------------------------------
# SCENARIO 1: The 'while' loop (Reading streams)
# ---------------------------------------------------------
# We assign 'data' and check if it's empty in one clean step.
# (Assume 'file_stream' is an opened file object)

# while (data := file_stream.read(1024)) != "":
#     print(f"Processing chunk: {data}")


# ---------------------------------------------------------
# SCENARIO 2: List Comprehensions
# ---------------------------------------------------------

# BAD: The slow function runs TWICE per user (Once for 'if', once to save it).
# Takes ~4 seconds total.
bad_scores = [
    slow_database_query(user) 
    for user in users 
    if slow_database_query(user) > 15
]

# GOOD: The Walrus Operator saves the result in 'score' during the 'if' check.
# We reuse 'score' at the front of the comprehension. 
# Takes ~2 seconds total!
good_scores = [
    score 
    for user in users 
    if (score := slow_database_query(user)) > 15
]

print(good_scores) 
# Note: The variable 'score' has now "leaked" and still exists in memory here!